In [2]:
from transformers import AutoTokenizer, AutoModel
def inspect_sentence(text,tokenizer):
    encoded = tokenizer(text,truncation=True,padding = True, return_tensors="pt")
    return encoded
def inspect_model(encoded,model):
    outputs = model(**encoded)
    print(f"outputs : {outputs[0][0][:10]}")
    print(f"last_hidden_state_shape : {outputs.last_hidden_state.shape}")
    print(f"outputs Type : {type(outputs)}")

In [3]:
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
text = "I love learning Transformers!"
inspect_model(inspect_sentence(text,tokenizer),model)

outputs : tensor([[ 0.6031,  0.0210,  0.1191,  ..., -0.0704,  0.3448,  0.0084],
        [ 0.3324, -0.0397,  0.2901,  ...,  0.1657,  0.0360,  0.4170],
        [ 0.3275,  0.2188, -0.3740,  ...,  0.5549, -0.4262,  0.0958],
        ...,
        [ 0.2512,  0.3698,  0.1805,  ...,  0.2003,  0.1382, -0.1633],
        [ 0.9154,  0.0804,  0.6942,  ..., -0.0177, -0.0951, -0.1033],
        [ 0.6660,  0.7473,  0.1796,  ...,  0.0396,  0.6812, -0.6343]],
       grad_fn=<SliceBackward0>)
last_hidden_state_shape : torch.Size([1, 7, 768])
outputs Type : <class 'transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions'>


In [5]:
embedding_layer = model.embeddings.word_embeddings
encoded = inspect_sentence(text,tokenizer)
input_ids = encoded["input_ids"]
embeddings = model.embeddings(input_ids=input_ids)

print(embeddings.shape) # (Batch Size,Sequence Length, Feature Vector)

torch.Size([1, 7, 768])


In [6]:
import torch
Q_weights = torch.rand(768, 768)
K_weights = torch.rand(768, 768)
V_weights = torch.rand(768, 768)

In [7]:
def get_Q(embeddings,Q_weights):
    Q = embeddings @ Q_weights
    return Q

def get_K(embeddings,K_weights):
    K = embeddings @ K_weights
    return K

def get_V(embeddings,V_weights):
    V = embeddings @ V_weights
    return V



In [16]:
import numpy as np
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__ (self,d_model,head_dim):
        super().__init__()
        self.d_model = d_model
        self.d_k = head_dim
        self.Wq = torch.rand(self.d_model,self.d_k)
        self.Wk = torch.rand(self.d_model,self.d_k)
        self.Wv = torch.rand(self.d_model,self.d_k)

    def input_embeddings(self,embeddings):
        self.embeddings = embeddings

    def get_Q(self):
        Q = self.embeddings @ self.Wq
        return Q

    def get_K(self):
        K = self.embeddings @ self.Wk
        return K

    def get_V(self):
        V = self.embeddings @ self.Wv
        return V

    def AttentionScore(self,Q, K):
        Score = Q @ K.transpose(-2,-1) / np.sqrt(self.d_k)
        return Score

    def forward(self,embeddings):
        self.embeddings = embeddings
        Q = self.get_Q()
        K = self.get_K()
        V = self.get_V()
        m = nn.Softmax(dim=-1)
        attention_weights = m(self.AttentionScore(Q, K))
        output = attention_weights @ V
        return output, attention_weights

In [17]:
attention = SelfAttention(d_model=768, head_dim=64)
output, weights = attention.forward(embeddings)

print(output.shape)
print(weights.shape)

torch.Size([1, 7, 64])
torch.Size([1, 7, 7])


In [20]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.Wo = torch.rand(self.d_model,self.d_model)
        self.heads = []
        for i in range (self.num_heads):
            self.heads.append(SelfAttention(d_model=self.d_model, head_dim=self.head_dim))

    def forward(self, embeddings):
        outputs = []
        for head in self.heads:
            out, _ = head.forward(embeddings)
            outputs.append(out)

        concatenated = torch.cat(outputs, dim=-1)
        final = concatenated @ self.Wo
        return final

In [22]:
multi_attention = MultiHeadAttention(d_model=768, num_heads=12)
print(multi_attention.forward(embeddings))


tensor([[[-3219.2451, -3104.0078, -3154.3223,  ..., -3167.6123,
          -3075.8267, -3170.3271],
         [ 3637.9177,  3477.2749,  3557.5911,  ...,  3558.5918,
           3471.0024,  3579.9080],
         [ 3637.9172,  3477.2749,  3557.5906,  ...,  3558.5913,
           3471.0020,  3579.9077],
         ...,
         [-3149.3843, -3100.4126, -3111.3330,  ..., -3121.9841,
          -3009.5542, -3115.1763],
         [ 3608.5076,  3437.2104,  3521.1882,  ...,  3508.9175,
           3420.9692,  3544.6812],
         [ 3353.6558,  3223.6006,  3239.9451,  ...,  3268.2117,
           3216.1570,  3311.3748]]], grad_fn=<UnsafeViewBackward0>)


In [31]:
import math
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(p = 0.1),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.linear_relu_stack(x)


In [32]:
class Encoder(nn.Module):
    def __init__(self,d_model,num_heads,d_ff):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.pe = PositionalEncoding(d_model = self.d_model)
        self.attention = MultiHeadAttention(self.d_model,self.num_heads)
        self.ffn = FeedForward(self.d_model,self.d_ff)
        self.norm1 = nn.RMSNorm(d_model)
        self.norm2 = nn.RMSNorm(d_model)

    def forward(self,x):
        x = self.pe(x)
        attention = self.attention(x)
        x = self.norm1(x + attention)
        ffn = self.ffn(x)
        x = self.norm2(x + ffn)
        return x

tensor([[[ 0.4496,  1.0977, -0.2074,  ...,  1.0578,  0.0406,  0.9049],
         [-0.1998,  0.7796,  1.4897,  ...,  2.1994, -0.8031,  1.5556],
         [ 1.3178,  0.0777,  0.6749,  ...,  0.6742, -0.8980,  0.4830],
         ...,
         [-0.0453, -0.9820, -0.7825,  ...,  2.2872, -0.1411,  1.4429],
         [-1.5208,  1.4222, -0.8252,  ...,  1.8117, -0.7606,  0.7679],
         [-0.5113,  0.8496, -0.1668,  ...,  1.6098, -0.7143,  1.4329]]],
       grad_fn=<AddBackward0>)
